# Mora SP Cup 2026 — NAFNet training on Colab T4 (Google Drive)

**Before starting:** upload the whole `mora_sp_cup_2026` project folder (including
`competition_data/public/`) to **My Drive/Colab Notebooks**, so this path exists:
`My Drive/Colab Notebooks/mora_sp_cup_2026/scripts/train.py`

Then: **Runtime → Change runtime type → T4 GPU**, and **Runtime → Run all**.

Code and data are read directly from Drive. Checkpoints go to `My Drive/mora_run/`,
so training **resumes automatically** if Colab disconnects: just run all cells again.
When training finishes, the weights are copied to
`My Drive/Colab Notebooks/mora_sp_cup_2026/scripts/weights/nafnet_small.pt`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Settings

Change `PROJECT_DIR` if you put the folder somewhere else. The noisy and ground-truth folders
are found automatically inside `DATA_SEARCH_DIR`. If your dataset is in a different Drive folder,
for example a shortcut to the official dataset folder, set `DATA_SEARCH_DIR` to that folder.

In [ ]:
import os, re

PROJECT_DIR     = '/content/drive/MyDrive/Colab Notebooks/mora_sp_cup_2026'
DATA_SEARCH_DIR = f'{PROJECT_DIR}/competition_data/public'
RUN_DIR         = '/content/drive/MyDrive/mora_run'        # checkpoints / logs (persistent)
FT_DIR          = '/content/drive/MyDrive/mora_run_all'    # optional fine-tune on all images
CACHE_DIR       = '/content/cache'                         # fast local disk

assert os.path.isfile(f'{PROJECT_DIR}/scripts/train.py'), f'scripts/train.py not found in {PROJECT_DIR}'
assert os.path.isfile(f'{PROJECT_DIR}/evaluation/evaluate.py'), f'evaluation/evaluate.py not found in {PROJECT_DIR}'

def find_image_dir(root, pattern, prefer):
    # folder with the most files matching `pattern`, preferring names containing `prefer`
    best = None
    for d, _, files in os.walk(root):
        n = sum(bool(re.fullmatch(pattern, f)) for f in files)
        if n >= 400:
            key = (prefer in d.lower(), n)
            if best is None or key > best[0]:
                best = (key, d)
    return best[1] if best else None

NOISY_DIR = find_image_dir(DATA_SEARCH_DIR, r'\d{3}_noise\.png', 'noisy')
GT_DIR    = find_image_dir(DATA_SEARCH_DIR, r'\d{3}\.png', 'ground')
assert NOISY_DIR and GT_DIR, f'could not find the 460 noisy / ground-truth images under {DATA_SEARCH_DIR}'
print('noisy       :', NOISY_DIR, len(os.listdir(NOISY_DIR)), 'files')
print('ground truth:', GT_DIR, len(os.listdir(GT_DIR)), 'files')
os.makedirs(RUN_DIR, exist_ok=True)
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'    # no __pycache__ folders in your Drive project
os.chdir(PROJECT_DIR)

In [ ]:
!pip -q install opencv-python-headless scikit-image

## 2. Train

* `--iters 40000` is roughly 3 h on a T4. Check the `it/s` printed after the first minutes;
  if your Colab session is limited, lower `--iters` **before** starting (the cosine schedule depends on it).
* `--max_hours` saves and stops cleanly; re-running this cell resumes from `last.pt`.
* The first run spends ~10–15 min reading the images from Drive and building a pre-processed cache.

In [ ]:
os.chdir(PROJECT_DIR)
!python scripts/train.py \
    --noisy_dir "{NOISY_DIR}" \
    --gt_dir "{GT_DIR}" \
    --work_dir "{RUN_DIR}" \
    --cache_dir "{CACHE_DIR}" \
    --preset small --iters 40000 --batch 8 --crop 256 --final_crop 384 \
    --val_every 2500 --workers 2 --max_hours 11.5

In [ ]:
!grep "val" "{RUN_DIR}/log.txt" | tail -20

## 3. Optional: short fine-tune on all 460 images

A low-LR fine-tune that also includes the 40 validation images usually helps a little
on unseen images. Its validation numbers are no longer unbiased (those images are now
seen), so report the numbers from step 2 and keep `mora_run/nafnet_small.pt` as a backup.

In [ ]:
os.chdir(PROJECT_DIR)
!python scripts/train.py \
    --noisy_dir "{NOISY_DIR}" \
    --gt_dir "{GT_DIR}" \
    --work_dir "{FT_DIR}" \
    --cache_dir "{CACHE_DIR}" \
    --init "{RUN_DIR}/nafnet_small.pt" \
    --preset small --iters 8000 --lr 2e-4 --warmup 200 --final_frac 0 \
    --crop 384 --batch 4 --val_every 2000 --workers 2 --train_on_all

## 4. Save the weights into the project folder

Copies the chosen weights to `My Drive/Colab Notebooks/mora_sp_cup_2026/scripts/weights/nafnet_small.pt`
and also downloads them, so you can put the file at the same path on your PC.

In [ ]:
import shutil
USE_FINE_TUNED = False    # True to use the step-3 model

src = f'{FT_DIR if USE_FINE_TUNED else RUN_DIR}/nafnet_small.pt'
dst = f'{PROJECT_DIR}/scripts/weights/nafnet_small.pt'
os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copy(src, dst)
print('copied', src, '->', dst)

from google.colab import files
files.download(dst)